In [ ]:
import math
import statistics
import numpy as np
import pandas as pd
import matplotlib

In [ ]:
# Task 1 - Load and Inspect

df_orders = pd.DataFrame(pd.read_csv('orders.csv'))
df_customers = pd.DataFrame(pd.read_csv('customers.csv'))
df_products = pd.DataFrame(pd.read_csv('products.csv'))

df_orders.shape

(180, 9)

In [ ]:
# Task 2 - Standardize payment method casing

df_orders['payment_method'].unique()

In [ ]:
df_orders['payment_method'] = df_orders['payment_method'].str.upper()
display(df_orders['payment_method'].value_counts())

,count
payment_method,
CARD,70
UPI,55
COD,55


In [ ]:
# Task 3 - Removing Duplicates

subset_cols = ['customer_id', 'product_id', 'order_date', 'quantity', 'discount_pct', 'payment_method', 'rating', 'returned']

# Identify duplicate rows (excluding the first occurrence)
duplicates = df_orders[df_orders.duplicated(subset=subset_cols, keep='first')]

# Drop the duplicates from the dataset
orders_clean = df_orders.drop_duplicates(subset=subset_cols, keep='first')

# Print the 5 dropped order_id values
print("Dropped order_id values:")
print(duplicates['order_id'].head(5).tolist())

# Check the new shape
print(f"Cleaned shape: {orders_clean.shape}")

Dropped order_id values:
['O0176', 'O0177', 'O0178', 'O0179', 'O0180']
Cleaned shape: (175, 9)


In [ ]:
# Task 4 — Impute missing values

# Count the number of missing values before imputation
null_discount_count = orders_clean['discount_pct'].isnull().sum()
null_rating_count = orders_clean['rating'].isnull().sum()

print(f"{null_discount_count} rows affected for discount_pct.")

# Calculate and print the median rating before imputation
median_rating = orders_clean['rating'].median()
print(f"Median rating of the non-null ratings: {median_rating}")
print(f"{null_rating_count} rows affected for rating.")

# Impute missing values securely
orders_clean = orders_clean.copy()
orders_clean['discount_pct'] = orders_clean['discount_pct'].fillna(0.0)
orders_clean['rating'] = orders_clean['rating'].fillna(median_rating)

# Verify final null counts
print("\nRemaining missing values after imputation:")
print(orders_clean[['discount_pct', 'rating']].isnull().sum().to_dict())

display(orders_clean.head())

In [ ]:
# Task 5 — Merge and reconcile against Part 1

# 1. Merge the cleaned orders with products (on product_id) and customers (on customer_id)
merged_clean = orders_clean.merge(df_products, on='product_id', how='left')
merged_clean = merged_clean.merge(df_customers, on='customer_id', how='left')

# Compute order_value per row on cleaned frame
merged_clean['order_value'] = merged_clean['quantity'] * merged_clean['price'] * (1 - merged_clean['discount_pct'] / 100.0)
total_clean_val = merged_clean['order_value'].sum()
print(f"Total order value across 175 cleaned rows: ₹{total_clean_val:,.2f}")

# 2. Calculate the order value of the 5 dropped duplicate rows to verify the delta
merged_duplicates = duplicates.merge(df_products, on='product_id', how='left')
merged_duplicates['order_value'] = merged_duplicates['quantity'] * merged_duplicates['price'] * (1 - merged_duplicates['discount_pct'] / 100.0)
total_dup_val = merged_duplicates['order_value'].sum()
print(f"Sum of order values for the 5 dropped duplicate rows: ₹{total_dup_val:,.2f}")

# 3. Print the one-paragraph reconciliation note
print("""
Reconciliation Note:
The total order value of the cleaned dataset (₹97,358.30 across 175
rows) is exactly ₹2,501.90 lower than the Part 1 Report (a)'s raw total
of ₹99,860.20. This delta is entirely and explicitly attributed to
the 5 duplicate rows (order IDs: 'O0176', 'O0177', 'O0178', 'O0179',
'O0180') removed during deduplication in Task 3, whose combined order
value sums to exactly ₹2,501.90. The discount and rating imputations
performed in Task 4 did not alter the order values, as the imputed
discount_pct of 0% corresponds to zero actual promotional discount
(preserving the base value), and the rating imputation has no impact
on revenue calculations.
""")

In [ ]:
# Task 6 — IQR outlier detection on quantity

# Compute Q1, Q3, IQR, lower and upper bounds for quantity
Q1 = merged_clean['quantity'].quantile(0.25)
Q3 = merged_clean['quantity'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(f"Q1 = {Q1}")
print(f"Q3 = {Q3}")
print(f"IQR = {IQR}")
print(f"Lower bound = {lower_bound}")
print(f"Upper bound = {upper_bound}")

# Identify outlier rows
outliers_mask = (merged_clean['quantity'] < lower_bound) | (merged_clean['quantity'] > upper_bound)
outliers = merged_clean[outliers_mask]

print(f"\nIdentified {len(outliers)} outlier rows:")
display(outliers[['order_id', 'quantity']])

# Flag outliers in both frames with 'is_outlier'
merged_clean['is_outlier'] = outliers_mask
orders_clean['is_outlier'] = orders_clean['order_id'].isin(outliers['order_id'])

In [ ]:
# Task 7 — Hypothesis: does COD have a higher return rate?

# State the hypothesis explicitly
print("Hypothesis: Cash on Delivery (COD) orders have a significantly higher return rate compared to electronic payment methods (CARD, UPI).\n")

# Group by payment_method and calculate count and mean of returns
hypothesis_test = merged_clean.groupby('payment_method')['returned'].agg(['count', 'mean']).reset_index()

# Rename columns for clarity
hypothesis_test.columns = ['Payment Method', 'Total Orders', 'Return Rate']

# Convert Return Rate to percentage
hypothesis_test['Return Rate (%)'] = (hypothesis_test['Return Rate'] * 100).round(1)

# Display the results
display(hypothesis_test)

# Print verified rates and validation flag
print("\nVerified Return Rates:")
for idx, row in hypothesis_test.iterrows():
    print(f"{row['Payment Method']}: {row['Return Rate (%)']}%")

print("\nConclusion: The hypothesis is CONFIRMED. COD has a return rate of 44.4%, which is vastly higher than CARD (14.7%) and UPI (18.9%).")

In [ ]:
# Task 8 — Multi-level segmentation

# Group by payment_method and city_tier, calculating counts and return rates
segment_analysis = merged_clean.groupby(['payment_method', 'city_tier'])['returned'].agg(['count', 'mean']).reset_index()
segment_analysis.columns = ['payment_method', 'city_tier', 'order_count', 'return_rate']
segment_analysis['return_rate_pct'] = (segment_analysis['return_rate'] * 100).round(1)

print("Multi-level Segmentation (Payment Method + City Tier):\n")
display(segment_analysis)

# Separate and show the comparison within COD orders
cod_t1 = segment_analysis[(segment_analysis['payment_method'] == 'COD') & (segment_analysis['city_tier'] == 1)].iloc[0]
cod_t2 = segment_analysis[(segment_analysis['payment_method'] == 'COD') & (segment_analysis['city_tier'] == 2)].iloc[0]

print("\nCOD Risk Breakdown:")
print(f"- COD Tier-1 cities: {cod_t1['order_count']} orders with a return rate of {cod_t1['return_rate_pct']}%")
print(f"- COD Tier-2 cities: {cod_t2['order_count']} orders with a return rate of {cod_t2['return_rate_pct']}%")

print("\nExplicit Finding:\nThe single highest-risk segment is COD + Tier-2 cities, which has a critical return rate of 54.5%. This is significantly higher than COD + Tier-1 cities (37.5%), proving that the blended COD return rate of 44.4% hides how concentrated the delivery risk is in Tier-2 urban segments.")

In [ ]:
# Task 9 — Correlation analysis

# Compute correlation matrix
cols_to_correlate = ['rating', 'returned', 'discount_pct', 'quantity']
corr_matrix = merged_clean[cols_to_correlate].corr()

print("Correlation Matrix:")
display(corr_matrix.round(4))

def get_strength_band(val):
    abs_val = abs(val)
    if abs_val < 0.2:
        return "Negligible (|r| < 0.2)"
    elif abs_val < 0.4:
        return "Weak (0.2 <= |r| < 0.4)"
    elif abs_val < 0.7:
        return "Moderate (0.4 <= |r| < 0.7)"
    else:
        return "Strong (0.7 <= |r| <= 1.0)"

# Evaluate all 6 pairwise correlations
pairs = [
    ('rating', 'returned'),
    ('rating', 'discount_pct'),
    ('rating', 'quantity'),
    ('returned', 'discount_pct'),
    ('returned', 'quantity'),
    ('discount_pct', 'quantity')
]

print("\nPairwise Correlation Analysis:")
for col1, col2 in pairs:
    r_val = corr_matrix.loc[col1, col2]
    band = get_strength_band(r_val)
    print(f"- {col1} vs {col2}: r = {r_val:+.4f} -> Strength: {band}")

# Explicit evaluation of the hypothesis
discount_returned_corr = corr_matrix.loc['discount_pct', 'returned']
print(f"\nHypothesis Evaluation:\n"
      f"The hypothesis that 'higher discounts reduce returns' is BUSTED. "
      f"The correlation between discount_pct and returned is approximately {discount_returned_corr:.4f}, "
      f"which falls squarely into the Negligible correlation band (|r| < 0.2).")

In [ ]:
# Task 10 — Outlier-corrected time series

# Convert order_date to datetime and extract year-month
merged_clean['order_date'] = pd.to_datetime(merged_clean['order_date'])
merged_clean['year_month'] = merged_clean['order_date'].dt.to_period('M')

# Scenario 1: Including all cleaned rows (including outliers)
monthly_with_outliers = merged_clean.groupby('year_month')['order_value'].sum()

# Scenario 2: Excluding Task 6 outlier orders
monthly_without_outliers = merged_clean[~merged_clean['is_outlier']].groupby('year_month')['order_value'].sum()

# Display both series
print("Scenario 1: Monthly total order_value (including outliers):")
for month, value in monthly_with_outliers.items():
    print(f"  {month}: {value:.2f}")

print("\nScenario 2: Outlier-corrected monthly total order_value (excluding outliers):")
for month, value in monthly_without_outliers.items():
    print(f"  {month}: {value:.2f}")

# Print explicit findings
print("""
Finding:
January's apparent sales lead (₹29,582.10) is entirely an artifact of the two bulk orders landing in January (order O0011 with a quantity of 25 on 2026-01-28, and order O0098 with a quantity of 30 on 2026-01-10). Once these two outliers are excluded, the outlier-corrected trend shows that March is the genuine peak month of sales (₹20,318.90), compared to an outlier-corrected total of just ₹11,637.10 in January.
""")

In [ ]:
# Task 11 — Two visualizations (analysis/visualize.py)
import os
import matplotlib.pyplot as plt

# Create directory if it doesn't exist
os.makedirs('visualizations', exist_ok=True)

# --- Visualization 1: Return Rate by Payment Method ---
# Sort descending
hypothesis_sorted = hypothesis_test.sort_values(by='Return Rate (%)', ascending=False)

plt.figure(figsize=(8, 5))
bars = plt.bar(
    hypothesis_sorted['Payment Method'],
    hypothesis_sorted['Return Rate (%)'],
    color=['#e74c3c', '#3498db', '#2ecc71'],
    edgecolor='black'
)

# Add exact percentage labels on top of each bar
for bar in bars:
    yval = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width()/2.0,
        yval + 1,
        f"{yval:.1f}%",
        ha='center',
        va='bottom',
        fontweight='bold'
    )

plt.title("COD Returns at 44.4% — 3x Card", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Payment Method", fontsize=11)
plt.ylabel("Return Rate (%)", fontsize=11)
plt.ylim(0, 50)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig('visualizations/return_rate_by_payment.png', dpi=150)
plt.show()
plt.close()


# --- Visualization 2: Outlier-Corrected Monthly Revenue Trend ---
# Convert Period Index to String for plotting
months_str = [str(m) for m in monthly_without_outliers.index]
values = monthly_without_outliers.values

plt.figure(figsize=(10, 5))
plt.plot(
    months_str,
    values,
    marker='o',
    linewidth=2.5,
    color='#2c3e50',
    markerfacecolor='#e74c3c',
    markersize=8
)

# Annotate the peak month (March)
for i, (m, v) in enumerate(zip(months_str, values)):
    if m == '2026-03':
        plt.annotate(
            f"Peak: ₹{v:,.2f}",
            xy=(m, v),
            xytext=(i, v + 1000),
            arrowprops=dict(facecolor='black', shrink=0.08, width=1, headwidth=6),
            fontweight='bold',
            ha='center'
        )
    else:
        plt.text(i, v - 1000, f"₹{v:,.0f}", ha='center', va='top', fontsize=9, color='#555555')

plt.title("Outlier-Corrected Monthly Revenue (True Peak: March 2026)", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Month (Year-Month)", fontsize=11)
plt.ylabel("Total Order Value (₹)", fontsize=11)
plt.ylim(5000, 25000)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('visualizations/monthly_revenue_trend.png', dpi=150)
plt.show()
plt.close()

PART 3 : GenAI Powered Insight Narrator

In [ ]:
# Task 1 — narrator/findings.json

import json
import os

# Create the narrator directory if it doesn't exist
os.makedirs('narrator', exist_ok=True)

# Structuring the data dynamically from previous run outputs to prevent drift
findings_data = {
    "cleaned_total_revenue_inr": round(float(total_clean_val), 2),
    "raw_total_revenue_inr": round(float(total_clean_val + total_dup_val), 2),
    "duplicate_reconciliation_delta_inr": round(float(total_dup_val), 2),
    "return_rate_by_payment": {
        "COD": 44.4,
        "CARD": 14.7,
        "UPI": 18.9
    },
    "highest_risk_segment": {
        "payment_method": "COD",
        "city_tier": 2,
        "return_rate_pct": 54.5
    },
    "true_peak_month": {
        "month": "2026-03",
        "revenue_inr": round(float(monthly_without_outliers['2026-03']), 2)
    },
    "outlier_inflated_month": {
        "month": "2026-01",
        "apparent_revenue_inr": round(float(monthly_with_outliers['2026-01']), 2),
        "corrected_revenue_inr": round(float(monthly_without_outliers['2026-01']), 2)
    }
}

# Write the structured JSON file
file_path = 'narrator/findings.json'
with open(file_path, 'w') as f:
    json.dump(findings_data, f, indent=4)

print(f"Successfully wrote findings to {file_path}:")
print(json.dumps(findings_data, indent=4))

import shutil

# Copy the file to the root content directory to make it immediately visible in the files explorer
shutil.copy('narrator/findings.json', 'findings.json')
print("Successfully copied 'narrator/findings.json' to the root directory as '/content/findings.json'!")
